In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from src.utils.preprocessing import wrangle_data
from sklearn.preprocessing import FunctionTransformer
from src.models.evaluate import evaluate, evaluate_anomaly, print_final_results, identify_best_model
from src.models.weighted_hybrid_model import train as train_hybrid_model
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score

In [3]:
# wrangle data
df = wrangle_data(False)

In [4]:
df.head(5)

,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_AMOUNT,IS_FRAUD,SENDER_TX_BEHAVIOR_ID,RECEIVER_INIT_BALANCE,RECEIVER_TX_BEHAVIOR_ID,SENDER_STEP_COUNT,RECEIVER_STEP_COUNT,SENDER_RECEIVER_PAIR_COUNT
0,6456,9069,6.144293,0,3,132.83,1,0.693147,1.609438,4.962845
1,7516,9543,6.337958,0,3,493.51,3,0.693147,1.098612,4.969813
2,2445,9356,6.396830,0,3,274.89,2,0.693147,0.693147,4.969813
3,2576,4617,6.146479,0,3,164.60,4,0.693147,1.098612,4.969813
4,3524,1773,6.007904,0,3,196.19,4,0.693147,0.693147,4.962845


In [5]:
# prepare features and target
X = df.drop(columns=["IS_FRAUD"])
y = df["IS_FRAUD"]

In [6]:
#split data into train, validation and test using temporal split
cutoff = int(len(X) * 0.8)
X_train_full, y_train_full = X.iloc[: cutoff], y.iloc[:cutoff]
X_test, y_test =  X.iloc[cutoff: ], y.iloc[cutoff:]

cutoff_train_val = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full.iloc[:cutoff_train_val], y_train_full.iloc[:cutoff_train_val]
X_validation, y_validation = X_train_full.iloc[cutoff_train_val:], y_train_full.iloc[cutoff_train_val:]


In [7]:
#X = 1048575
print("X length " + str(len(X)))
print("X_train length " +str(len(X_train)) + ", y_train length " +str(len(y_train)))
print("X_val length " +str(len(X_validation)) + ", y_val length " +str(len(y_validation)))
print("X_test length " +str(len(X_test))  + ", y_test length " +str(len(y_test)))

X length 1048575
X_train length 671088, y_train length 671088
X_val length 167772, y_val length 167772
X_test length 209715, y_test length 209715


In [8]:
random_forest_standard_pipeline = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])
random_forest_standard_pipeline.fit(X_train, y_train)

,steps,"[('smote', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](9,)","['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID','TX_AMOUNT',..., 'SENDER_STEP_COUNT','RECEIVER_STEP_COUNT','SENDER_RECEIVER_PAIR_COUNT']"
n_features_in_,int,9
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5


In [9]:
# train isolation forest
CONTAMINATION = 0.0013  # observed fraud rate of 0.13%
iso_model = IsolationForest(contamination=CONTAMINATION, random_state=42, n_jobs=-1)
iso_model.fit(X_train)

,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.0013
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [10]:
#isolation forest predictions
iso_model.predict(X_train)

array([1, 1, 1, ..., 1, 1, 1], shape=(671088,))

In [11]:
# Invert the anomaly scores for training data to make them more intuitive (higher score = more anomalous)
inverted_iso_model_training_score = -iso_model.decision_function(X_train)

In [12]:
hybrid_weighted_average_model = train_hybrid_model(random_forest_standard_pipeline, iso_model, X_validation, y_validation)

Selected weight = 0.95  (validation PR-AUC = 0.9937)


In [13]:
X_train_with_iso_score = X_train.copy()
X_train_with_iso_score["ISO_SCORE"] = inverted_iso_model_training_score
X_train_with_iso_score.head(5)

,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_AMOUNT,SENDER_TX_BEHAVIOR_ID,RECEIVER_INIT_BALANCE,RECEIVER_TX_BEHAVIOR_ID,SENDER_STEP_COUNT,RECEIVER_STEP_COUNT,SENDER_RECEIVER_PAIR_COUNT,ISO_SCORE
0,6456,9069,6.144293,3,132.83,1,0.693147,1.609438,4.962845,-0.137699
1,7516,9543,6.337958,3,493.51,3,0.693147,1.098612,4.969813,-0.171347
2,2445,9356,6.396830,3,274.89,2,0.693147,0.693147,4.969813,-0.143308
3,2576,4617,6.146479,3,164.60,4,0.693147,1.098612,4.969813,-0.159178
4,3524,1773,6.007904,3,196.19,4,0.693147,0.693147,4.962845,-0.131139


In [14]:
random_forest_feature_level_fusion_pipeline = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])
random_forest_feature_level_fusion_pipeline.fit(X_train_with_iso_score, y_train)

,steps,"[('smote', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](10,)","['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID','TX_AMOUNT',..., 'RECEIVER_STEP_COUNT','SENDER_RECEIVER_PAIR_COUNT','ISO_SCORE']"
n_features_in_,int,10
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5


In [15]:
# Invert the anomaly scores for validation data to make them more intuitive (higher score = more anomalous)
inverted_iso_model_validation_score = -iso_model.decision_function(X_validation)
X_validation_with_iso_score = X_validation.copy()
X_validation_with_iso_score["ISO_SCORE"] = inverted_iso_model_validation_score
X_validation_with_iso_score.head(5)

,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_AMOUNT,SENDER_TX_BEHAVIOR_ID,RECEIVER_INIT_BALANCE,RECEIVER_TX_BEHAVIOR_ID,SENDER_STEP_COUNT,RECEIVER_STEP_COUNT,SENDER_RECEIVER_PAIR_COUNT,ISO_SCORE
671088,4453,9576,6.194181,3,409.75,3,0.693147,1.945910,4.969813,-0.155813
671089,2290,9752,12.683550,2,115.98,4,0.693147,1.609438,1.098612,0.006946
671090,9424,4294,6.004924,3,232.77,2,0.693147,0.693147,4.969813,-0.140872
671091,8429,7351,6.119220,3,282.46,2,0.693147,1.098612,4.969813,-0.170344
671092,8569,8773,6.284806,3,190.53,4,0.693147,1.386294,4.962845,-0.166207


In [16]:
class CascadeHybrid():
    """
    Cascade / two-stage fusion: Random Forest makes the decision by
    default; Isolation Forest is only consulted for samples whose
    RF-normalised score falls inside an uncertain band. Inherits score
    extraction and normalisation from WeightedAverageHybrid.
    """

    def __init__(self, classifier, isolation_forest, band_candidates=None):
        self.classifier = classifier
        self.isolation_forest = isolation_forest
        self.band_candidates = band_candidates or [
            (0.40, 0.60),
            (0.35, 0.65),
            (0.45, 0.55),
            (0.30, 0.70),
        ]
        
        self._clf_min = self._clf_max = None
        self._iso_min = self._iso_max = None
        self.best_lower_ = None
        self.best_upper_ = None
        self.best_pr_auc_ = None
        self.cascade_history_ = None

    def _raw_scores(self, X):
            clf_score = self.classifier.predict_proba(X)[:, 1]
            # IsolationForest.decision_function: higher = more normal, so flip sign
            iso_score = -self.isolation_forest.decision_function(X)
            return clf_score, iso_score
    
    def _normalise(self, clf_score, iso_score, fit_ranges=False):
            if fit_ranges:
                self._clf_min = clf_score.min()
                self._clf_max = clf_score.max()
                self._iso_min = iso_score.min()
                self._iso_max = iso_score.max()
    
            clf_norm = (clf_score - self._clf_min) / (self._clf_max - self._clf_min + 1e-12)
            iso_norm = (iso_score - self._iso_min) / (self._iso_max - self._iso_min + 1e-12)
            return clf_norm, iso_norm
    
    def fit_band(self, X_val, y_val):
        """
        Select the uncertain-score band that maximises PR-AUC on the
        validation set. Sets best_lower_, best_upper_, best_pr_auc_,
        and cascade_history_ as fitted attributes.
        """
        clf_score, iso_score = self._raw_scores(X_val)
        clf_norm, iso_norm = self._normalise(clf_score, iso_score, fit_ranges=True)
        # normalise scores to [0, 1] via min-max scaling using ranges fit on validation data

        best_lower, best_upper, best_pr_auc = None, None, -np.inf
        history = []

        for lower, upper in self.band_candidates:
            fused = clf_norm.copy()
            uncertain_mask = (clf_norm >= lower) & (clf_norm <= upper)
            fused[uncertain_mask] = iso_norm[uncertain_mask]

            pr_auc = average_precision_score(y_val, fused)
            history.append({
                "band": (round(float(lower), 2), round(float(upper), 2)),
                "pr_auc": pr_auc,
                "n_escalated": int(uncertain_mask.sum()),
            })

            if pr_auc > best_pr_auc:
                best_pr_auc, best_lower, best_upper = pr_auc, lower, upper

        self.best_lower_ = best_lower
        self.best_upper_ = best_upper
        self.best_pr_auc_ = best_pr_auc
        self.cascade_history_ = history

        print(f"Best band: ({best_lower}, {best_upper}), Best PR AUC: {best_pr_auc:.4f}")
        return self

    def transform_uncertain_predictions(self, X):
        """
        Apply the fitted cascade band to new data (e.g. test set).
        Reuses the min-max ranges fit during fit() — does not refit
        normalisation ranges on X, to avoid test-set leakage.
        """
        if self.best_lower_ is None:
            raise ValueError("CascadeHybrid instance is not fitted yet. Call 'fit' first.")

        clf_score, iso_score = self._raw_scores(X)
        clf_norm, iso_norm = self._normalise(clf_score, iso_score, fit_ranges=False)

        fused = clf_norm.copy()
        uncertain_mask = (clf_norm >= self.best_lower_) & (clf_norm <= self.best_upper_)
        fused[uncertain_mask] = iso_norm[uncertain_mask]

        return fused

    def predict_proba(self, X):
        """
        sklearn-compatible wrapper around transform(), so this class
        can be evaluated with the same evaluate() function used for
        the other models. Returns shape (n_samples, 2): [P(legit), P(fraud)].
        """
        fraud_score = self.transform_uncertain_predictions(X)
        return np.column_stack([1 - fraud_score, fraud_score])

    def predict(self, X, threshold=0.6):
        # set threhold to 0.6 as seen in the validation set, the best band is (0.4, 0.6), so we can use 0.6 as the threshold for predicting fraud
        fraud_score = self.transform_uncertain_predictions(X)
        return (fraud_score > threshold).astype(int) #using greater than since less tahn or  equal to is considered n higher band during fitting

In [17]:
class ThresholdHybrid:
    """
    Threshold-based (OR-rule) fusion: flag as fraud if EITHER the
    classifier or Isolation Forest exceeds its own tuned threshold.
    Simple, raises recall, costs precision compared to cascade or
    feature-level fusion — but a defensible baseline fallback.

    Threshold pair selected via F1-score on the validation set, since
    the OR-rule's operating point is defined by the thresholds
    themselves (an "operationally meaningful threshold", per
    supervisor guidance) — not by a threshold-free ranking metric.
    """

    def __init__(self, classifier, isolation_forest):
        self.classifier = classifier
        self.isolation_forest = isolation_forest

        self._clf_min = self._clf_max = None
        self._iso_min = self._iso_max = None
        self.best_clf_threshold_ = None
        self.best_iso_threshold_ = None
        self.best_f1_ = None
        self.threshold_history_ = None

    def _raw_scores(self, X):
        clf_score = self.classifier.predict_proba(X)[:, 1]
        # IsolationForest.decision_function: higher = more normal, so flip sign
        iso_score = -self.isolation_forest.decision_function(X)
        return clf_score, iso_score

    def _normalise(self, clf_score, iso_score, fit_ranges=False):
        if fit_ranges:
            self._clf_min = clf_score.min()
            self._clf_max = clf_score.max()
            self._iso_min = iso_score.min()
            self._iso_max = iso_score.max()

        clf_norm = (clf_score - self._clf_min) / (self._clf_max - self._clf_min + 1e-12)
        iso_norm = (iso_score - self._iso_min) / (self._iso_max - self._iso_min + 1e-12)
        return clf_norm, iso_norm

    def fit_thresholds(self, X_val, y_val, thresholds=None):
        """
        Grid-search over (clf_threshold, iso_threshold) pairs, selecting
        the combination that maximises F1 on the validation set under
        the OR-rule: flagged = (clf_norm >= clf_t) | (iso_norm >= iso_t).

        PR-AUC (continuous, via np.maximum of the two normalised scores)
        is logged per grid point for reference, but does not drive
        selection here — it is threshold-independent and therefore
        doesn't reflect this strategy's specific operating point.
        """
        if thresholds is None:
            thresholds = np.arange(0.0, 1.01, 0.05)

        clf_score, iso_score = self._raw_scores(X_val)
        clf_norm, iso_norm = self._normalise(clf_score, iso_score, fit_ranges=True)

        # Continuous ranking score, independent of any threshold pair —
        # computed once, logged per row for reference only.
        #continuous_pr_auc = average_precision_score(y_val, np.maximum(clf_norm, iso_norm))

        best_clf_t, best_iso_t, best_f1 = None, None, -np.inf
        history = []

        for clf_t in thresholds:
            for iso_t in thresholds:
                y_pred = ((clf_norm >= clf_t) | (iso_norm >= iso_t)).astype(int)
                f1 = f1_score(y_val, y_pred, zero_division=0)

                history.append({
                    "clf_threshold": round(float(clf_t), 2),
                    "iso_threshold": round(float(iso_t), 2),
                    "f1_score": f1,
                   # "pr_auc_continuous": continuous_pr_auc,
                    "n_flagged": int(y_pred.sum()),
                })

                if f1 > best_f1:
                    best_f1, best_clf_t, best_iso_t = f1, clf_t, iso_t

        self.best_clf_threshold_ = best_clf_t
        self.best_iso_threshold_ = best_iso_t
        self.best_f1_ = best_f1
        self.threshold_history_ = history

        return self

    def _scores(self, X):
        clf_score, iso_score = self._raw_scores(X)
        clf_norm, iso_norm = self._normalise(clf_score, iso_score, fit_ranges=False)
        return clf_norm, iso_norm

    def predict_proba(self, X):
        """
        No natural continuous score exists for an OR-rule, so the max
        of the two normalised scores is used as a ranking proxy for
        PR-AUC reporting in evaluate() — used for cross-strategy
        comparison but not for threshold selection.
        """
        clf_norm, iso_norm = self._scores(X)
        fraud_score = np.maximum(clf_norm, iso_norm)
        return np.column_stack([1 - fraud_score, fraud_score])

    def predict(self, X):
        if self.best_clf_threshold_ is None or self.best_iso_threshold_ is None:
            raise ValueError("ThresholdHybrid instance is not fitted yet. Call 'fit_thresholds' first.")
        clf_norm, iso_norm = self._scores(X)
        y_pred = ((clf_norm >= self.best_clf_threshold_) | (iso_norm >= self.best_iso_threshold_)).astype(int)
        return y_pred

In [18]:
class VotingHybrid:
    """
    Soft voting fusion: average the normalised classifier and Isolation
    Forest scores with equal (0.5 / 0.5) weight — an unweighted variant
    of weighted averaging, where no weight is tuned on validation data.
    The fused score is thresholded to produce the final decision, with
    the threshold tuned on F1 (the operationally meaningful metric),
    consistent with ThresholdHybrid.
    """

    def __init__(self, classifier, isolation_forest):
        self.classifier = classifier
        self.isolation_forest = isolation_forest

        self._clf_min = self._clf_max = None
        self._iso_min = self._iso_max = None
        self.best_threshold_ = None
        self.best_f1_ = None
        self.threshold_history_ = None

    def _raw_scores(self, X):
        clf_score = self.classifier.predict_proba(X)[:, 1]
        # IsolationForest.decision_function: higher = more normal, so flip sign
        iso_score = -self.isolation_forest.decision_function(X)
        return clf_score, iso_score

    def _normalise(self, clf_score, iso_score, fit_ranges=False):
        if fit_ranges:
            self._clf_min = clf_score.min()
            self._clf_max = clf_score.max()
            self._iso_min = iso_score.min()
            self._iso_max = iso_score.max()

        clf_norm = (clf_score - self._clf_min) / (self._clf_max - self._clf_min + 1e-12)
        iso_norm = (iso_score - self._iso_min) / (self._iso_max - self._iso_min + 1e-12)
        return clf_norm, iso_norm

    def _voted_score(self, clf_norm, iso_norm):
        # Equal-weight (soft voting) average — no tuned weight, unlike
        # WeightedAverageHybrid.
        return 0.5 * clf_norm + 0.5 * iso_norm

    def fit_threshold(self, X_val, y_val, thresholds=None):
        """
        Tune a single decision threshold on the soft-voted score,
        maximising F1 on the validation set. Voting weights
        are fixed at 0.5 for each participating model.
        """
        if thresholds is None:
            thresholds = np.arange(0.0, 1.01, 0.05)

        clf_score, iso_score = self._raw_scores(X_val)
        clf_norm, iso_norm = self._normalise(clf_score, iso_score, fit_ranges=True)
        voted_score = self._voted_score(clf_norm, iso_norm)

        best_t, best_f1 = None, -np.inf
        history = []

        for t in thresholds:
            y_pred = (voted_score >= t).astype(int)
            f1 = f1_score(y_val, y_pred, zero_division=0)
            history.append({
                "threshold": round(float(t), 2),
                "f1_score": f1,
                "n_flagged": int(y_pred.sum()),
            })
            if f1 > best_f1:
                best_f1, best_t = f1, t

        self.best_threshold_ = best_t
        self.best_f1_ = best_f1
        self.threshold_history_ = history

        print(f"Best threshold: {best_t:.2f}, Best F1: {best_f1:.4f}")
        return self

    def _scores(self, X):
        clf_score, iso_score = self._raw_scores(X)
        clf_norm, iso_norm = self._normalise(clf_score, iso_score, fit_ranges=False)
        return clf_norm, iso_norm

    def predict_proba(self, X):
        """
        Returns the soft-voted (equal-weight average) score as the
        continuous ranking output — used for PR-AUC comparison
        in evaluate().
        """
        clf_norm, iso_norm = self._scores(X)
        fraud_score = self._voted_score(clf_norm, iso_norm)
        return np.column_stack([1 - fraud_score, fraud_score])

    def predict(self, X):
        if self.best_threshold_ is None:
            raise ValueError("VotingHybrid instance is not fitted yet. Call 'fit_threshold' first.")
        clf_norm, iso_norm = self._scores(X)
        voted_score = self._voted_score(clf_norm, iso_norm)
        return (voted_score >= self.best_threshold_).astype(int)

In [19]:
# executing the cascade model
cascade_model = CascadeHybrid(random_forest_standard_pipeline, iso_model)
cascade_model.fit_band(X_validation, y_validation)

cascade_test_scores = cascade_model.transform_uncertain_predictions(X_test)
cascade_test_pr_auc = average_precision_score(y_test, cascade_test_scores)
# cascade_test_f1 = f1_score(y_test, (cascade_test_scores >= 0.5).astype(int))

# print(f"Cascade test PR-AUC: {cascade_test_pr_auc:.4f}, F1: {cascade_test_f1:.4f}")
print(f"Cascade test PR-AUC: {cascade_test_pr_auc:.4f}")
# # Inspect the fitted band and validation history for the write-up
# print(cascade_model.best_lower_, cascade_model.best_upper_, cascade_model.best_pr_auc_)
# print(cascade_model.cascade_history_)

Best band: (0.4, 0.6), Best PR AUC: 0.9937
Cascade test PR-AUC: 0.9963


In [20]:
# executing the threshold-based hybrid model
threshold_model = ThresholdHybrid(random_forest_standard_pipeline, iso_model)
# Tune on F1 (default)
threshold_model.fit_thresholds(X_validation, y_validation)
print(f"Best thresholds: clf={threshold_model.best_clf_threshold_:.2f}, iso={threshold_model.best_iso_threshold_:.2f}, Best F1: {threshold_model.best_f1_:.4f}")


Best thresholds: clf=0.10, iso=1.00, Best F1: 0.9938


In [21]:
#executing the voting-based hybrid model
voting_model = VotingHybrid(random_forest_standard_pipeline, iso_model)
voting_model.fit_threshold(X_validation, y_validation)

Best threshold: 0.50, Best F1: 0.9938


## Empirical model selection

All hybrid approaches are evaluated using the same validation metrics: PR-AUC for ranking quality and F1-score at the fitted operational threshold for classification quality. The selection protocol is fixed before test evaluation: select the highest validation PR-AUC, using validation F1-score as the tie-breaker. The test set remains untouched until the selected model is evaluated once.

In [35]:
# Evaluate every approach on validation data using the same metrics.
validation_results = []
validation_results.append(evaluate(hybrid_weighted_average_model, X_validation, y_validation, "Weighted Average"))
validation_results.append(evaluate(random_forest_feature_level_fusion_pipeline, X_validation_with_iso_score, y_validation, "Feature-Level Fusion"))
validation_results.append(evaluate(cascade_model, X_validation, y_validation, "Cascade"))
validation_results.append(evaluate(threshold_model, X_validation, y_validation, "Threshold OR-Rule"))
validation_results.append(evaluate(voting_model, X_validation, y_validation, "Soft Voting"))

validation_metrics = pd.DataFrame(validation_results).sort_values(
    ["pr_auc", "f1_score"], ascending=False
 )
display(validation_metrics[["model", "pr_auc", "f1_score", "recall", "fpr", "fnr"]])

# Selection rule fixed before test evaluation: PR-AUC first, F1-score tie-breaker.
selected_model_name = validation_metrics.iloc[0]["model"]
selected_model = {
    "Random Forest (Standard)": random_forest_standard_pipeline,
    "Isolation Forest": iso_model,
    "Weighted Average": hybrid_weighted_average_model,
    "Feature-Level Fusion": random_forest_feature_level_fusion_pipeline,
    "Cascade": cascade_model,
    "Threshold OR-Rule": threshold_model,
    "Soft Voting": voting_model,
}[selected_model_name]
print(f"Selected using validation data: {selected_model_name}")

,model,pr_auc,f1_score,recall,fpr,fnr
0,Weighted Average,0.9937,0.9938,0.9877,0.0,0.0123
2,Cascade,0.9937,0.9907,0.9815,0.0,0.0185
1,Feature-Level Fusion,0.9919,0.9907,0.9815,0.0,0.0185
4,Soft Voting,0.9877,0.9938,0.9877,0.0,0.0123
3,Threshold OR-Rule,0.8999,0.9938,0.9877,0.0,0.0123


Selected using validation data: Weighted Average


In [34]:
identify_best_model(validation_results, sort_by_performance= True)



── Model Comparison ─────────────────────────────────────
                      precision  recall  f1_score  fpr     fnr  roc_auc  pr_auc
model                                                                          
Weighted Average            1.0  0.9877    0.9938  0.0  0.0123   0.9983  0.9937
Cascade                     1.0  0.9815    0.9907  0.0  0.0185   0.9969  0.9937
Feature-Level Fusion        1.0  0.9815    0.9907  0.0  0.0185   1.0000  0.9919
Soft Voting                 1.0  0.9877    0.9938  0.0  0.0123   0.9964  0.9877
Threshold OR-Rule           1.0  0.9877    0.9938  0.0  0.0123   0.9938  0.8999

── Best Model ───────────────────────────────────────────
  By PR-AUC   : Weighted Average          (0.9937)
  By F1-Score : Weighted Average          (0.9938)

  Overall best (PR-AUC + F1): Weighted Average


'Weighted Average'

In [22]:
# Invert the anomaly scores for test data to make them more intuitive (higher score = more anomalous)
inverted_iso_model_test_score = -iso_model.decision_function(X_test)
X_test_with_iso_score = X_test.copy()
X_test_with_iso_score["ISO_SCORE"] = inverted_iso_model_test_score
X_test_with_iso_score.head(5)

,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_AMOUNT,SENDER_TX_BEHAVIOR_ID,RECEIVER_INIT_BALANCE,RECEIVER_TX_BEHAVIOR_ID,SENDER_STEP_COUNT,RECEIVER_STEP_COUNT,SENDER_RECEIVER_PAIR_COUNT,ISO_SCORE
838860,7613,8164,4.903347,4,149.71,1,2.639057,0.693147,2.833213,-0.179320
838861,7613,2192,4.903347,4,191.36,1,2.639057,1.386294,2.833213,-0.140792
838862,7613,3625,4.903347,4,177.57,4,2.639057,0.693147,2.833213,-0.198492
838863,7613,5853,4.903347,4,348.35,5,2.639057,0.693147,2.833213,-0.196615
838864,7613,5941,4.903347,4,230.37,5,2.639057,0.693147,2.833213,-0.200098


In [ ]:
# Evaluate all hybrid models using PR-AUC and F1-Score

In [28]:
# Evaluate only the model selected using validation data.
# The test set is used once here for the final unbiased estimate.
final_test_results = []  
selected_X_test = (
    X_test_with_iso_score
    if selected_model_name == "Hybrid (Feature-Level Fusion)"
    else X_test
 )
final_test_results.append(evaluate(random_forest_standard_pipeline, selected_X_test, y_test, "Random Forest (Standard)"))
final_test_results.append(evaluate_anomaly(iso_model, selected_X_test, y_test, "Isolation Forest"))
final_test_results.append(evaluate(selected_model, selected_X_test, y_test, selected_model_name))
print_final_results(final_test_results)



── Model Comparison ─────────────────────────────────────
                           precision  recall  f1_score     fpr     fnr  roc_auc  pr_auc
model                                                                                  
Random Forest (Standard)      1.0000  0.9891    0.9945  0.0000  0.0109   0.9982  0.9963
Isolation Forest              0.0007  0.0036    0.0012  0.0064  0.9964   0.9335  0.0115
Hybrid (Weighted Average)     0.9927  0.9927    0.9927  0.0000  0.0073   0.9998  0.9964

── Best Model ───────────────────────────────────────────
  By PR-AUC   : Hybrid (Weighted Average) (0.9964)
  By F1-Score : Random Forest (Standard)  (0.9945)

  Final best (PR-AUC + F1): Random Forest (Standard)
